In [2]:
import torch 
import gymnasium as gym 
import matplotlib.pyplot as plt 
import cv2 as cv
import torch.nn as nn
import random
import numpy as np
from torch import optim
import copy
from scipy import stats
%matplotlib inline

In [3]:
env = gym.make('MountainCar-v0', render_mode="rgb_array")


In [4]:
obs, info = env.reset()


In [7]:
class QNetwork(nn.Module):
    def __init__(self, state_dim=2, action_dim=3, hidden_dim=64):
        super(QNetwork, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim)
        )
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')
    
    def forward(self, x):
        return self.net(x)

In [ ]:
class PrioritizedBuffer:
    def __init__(self, capacity, alpha=0.6):
        self.capacity = capacity
        self.alpha = alpha  
        self.buffer = []
        self.pos = 0
        self.priorities = np.zeros((capacity,), dtype=np.float32)
    
    def add(self, state, action, reward, next_state, done):
        max_prio = self.priorities.max() if self.buffer else 1.0
        
        if len(self.buffer) < self.capacity:
            self.buffer.append((state, action, reward, next_state, done))
        else:
            self.buffer[self.pos] = (state, action, reward, next_state, done)
        
        self.priorities[self.pos] = max_prio
        self.pos = (self.pos + 1) % self.capacity

    def sample(self, batch_size, beta=0.4):
        if len(self.buffer) == self.capacity:
            prios = self.priorities
        else:
            prios = self.priorities[:self.pos]
        probs = prios ** self.alpha
        probs /= probs.sum()
        indices = np.random.choice(len(self.buffer), batch_size, p=probs)
        samples = [self.buffer[idx] for idx in indices]
        total = len(self.buffer)
        weights = (total * probs[indices]) ** (-beta)
        weights /= weights.max() 
        states, actions, rewards, next_states, dones = zip(*samples)
        return (np.array(states), np.array(actions), np.array(rewards, dtype=np.float32), np.array(next_states), np.array(dones, dtype=np.float32), indices, torch.FloatTensor(weights))

    def update_priorities(self, batch_indices, batch_priorities):
        for idx, prio in zip(batch_indices, batch_priorities):
            self.priorities[idx] = prio + 1e-6


In [ ]:

def train_dqn_per(seed, num_episodes=500, rho=1):
    env = gym.make('MountainCar-v0')
    random.seed(seed) 
    np.random.seed(seed) 
    torch.manual_seed(seed)
    env = gym.wrappers.TimeLimit(env.env, max_episode_steps=2000)
    
    policy_net = QNetwork(2, 3); target_net = QNetwork(2, 3)
    target_net.load_state_dict(policy_net.state_dict())
    optimizer = optim.Adam(policy_net.parameters(), lr=1e-4) # PER often needs lower LR
    
    alpha, beta_start, beta_frames = 0.6, 0.4, 100000 
    buffer = PrioritizedBuffer(50000, alpha)
    
    gamma, batch_size, target_update_freq = 0.99, 64, 1000 
    eps, eps_min, eps_decay = 1.0, 0.01, 0.995
    total_steps, episode_returns = 0, []
    best_return = -2001

    for ep in range(num_episodes):
        state, _ = env.reset(seed=seed)
        ep_ret, done = 0, False
        
        while not done:
            if random.random() < eps:
                action = env.action_space.sample()
            else:
                with torch.no_grad():
                    action = policy_net(torch.FloatTensor(state)).argmax().item()
            
            next_s, rew, term, trunc, _ = env.step(action)
            done = term or trunc
            buffer.add(state, action, rew, next_s, term)
            state, ep_ret, total_steps = next_s, ep_ret + rew, total_steps + 1

            if len(buffer.buffer) > 1000:
                for _ in range(rho):
                    beta = min(1.0, beta_start + total_steps * (1.0 - beta_start) / beta_frames)
                    
                    s_batch, a_batch, r_batch, ns_batch, d_batch, indices, weights = buffer.sample(batch_size, beta)
                    
                    s_tensor = torch.FloatTensor(s_batch)
                    ns_tensor = torch.FloatTensor(ns_batch)
                    a_tensor = torch.LongTensor(a_batch).view(-1, 1)
                    r_tensor = torch.FloatTensor(r_batch).view(-1, 1)
                    d_tensor = torch.FloatTensor(d_batch).view(-1, 1)

                    current_q = policy_net(s_tensor).gather(1, a_tensor)
                    with torch.no_grad():
                        max_next_q = target_net(ns_tensor).max(1)[0].view(-1, 1)
                        target_q = r_tensor + (gamma * max_next_q * (1 - d_tensor))
                    
                    td_errors = torch.abs(target_q - current_q).detach().numpy()
                    
                    loss = (weights.view(-1, 1) * nn.MSELoss(reduction='none')(current_q, target_q)).mean()
                    
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()
                    
                    buffer.update_priorities(indices, td_errors.flatten())
            
            if total_steps % target_update_freq == 0:
                target_net.load_state_dict(policy_net.state_dict())
        
        eps = max(eps_min, eps * eps_decay)
        episode_returns.append(ep_ret)
        if ep_ret > best_return:
            best_return = ep_ret
            best_state_dict = copy.deepcopy(policy_net.state_dict()) 
            
    env.close()
    return episode_returns, policy_net.state_dict(), best_state_dict

In [ ]:
for i in range(15):
    returns, state_dict, best_dict = train_dqn_per(1, num_episodes=2)
    np.save(f"reurns_1_seed{i}_rho1_bonus.npy", returns)
    torch.save(state_dict, f"last_dict_seed{i}_rho1_bonus.pt")
    torch.save(best_dict, f"best_dict_seed{i}_rho1_bonus.pt")